In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)
from mcp import StdioServerParameters
os.makedirs("./memory", exist_ok=True)

In [2]:
params = {
    "command": "npx",
    "args": ["-y", "mcp-memory-libsql"],
    "env": {"LIBSQL_URL": f"file:{os.path.abspath('./memory/jf.db')}"}
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', title='Create new entities with observations', description='Create new entities with observations', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema=None, icons=None, annotations=None, meta=None),
 Tool(name='search_nodes', title='Search for entities and their relations using text search with relevance ranking', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'query': {'type': 'string'}, 'limit': {'type': 'number'}}, 'required': ['query']}, outputSchema=None, icons=None, 

In [3]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Jarif. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4o-mini"

In [4]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I've recorded the information about you, Jarif, and the MCP protocol. If you need to add more details or have specific questions, just let me know!

In [5]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Jarif. What do you know about me?")
    display(Markdown(result.final_output))

I know that your name is Jarif. Here are some details about you:

- You are an LLM engineer.
- You are teaching a course about AI Agents.
- You are involved with the MCP protocol. 

If you’d like to share more or update any information, feel free!

# The Second type of MCP server - runs locally, calls a web service

In [6]:
import os
print("TAVILY_API_KEY:", os.getenv("TAVILY_API_KEY")[:10] + "..." if os.getenv("TAVILY_API_KEY") else None)

TAVILY_API_KEY: tvly-dev-q...


In [7]:
remote_url = "https://mcp.tavily.com/mcp/?tavilyApiKey=" + os.getenv("TAVILY_API_KEY")

params = {
    "command": "npx",
    "args": ["-y", "mcp-remote", remote_url],
    "env": {}
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='tavily_search', title=None, description='Search the web for real-time information about any topic. Use this tool when you need up-to-date information that might not be available in your training data, or when you need to verify current facts. The search results will include relevant snippets and URLs from web pages. This is particularly useful for questions about current events, technology updates, or any topic that requires recent information.', inputSchema={'properties': {'query': {'description': 'Search query', 'type': 'string'}, 'max_results': {'default': 5, 'description': 'The maximum number of search results to return', 'type': 'integer'}, 'search_depth': {'default': 'basic', 'description': "The depth of the search. It can be 'basic' or 'advanced'", 'enum': ['basic', 'advanced'], 'type': 'string'}, 'topic': {'default': 'general', 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'enum': ['general', 'news'

In [8]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [9]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

As of late November 2025, Amazon's stock (AMZN) is experiencing a complex landscape. Here are the key takeaways from recent updates:

1. **Current Stock Performance**: Amazon shares recently closed at around $244.41, reflecting some volatility, especially following the earnings report in late October where the stock surged on strong AWS growth.

2. **Earnings and Growth**: The latest reporting highlighted a significant boost in Amazon's AWS cloud service, increasing sales by 20% year-over-year to $33 billion. This growth has been a crucial driver for investor confidence.

3. **Market Sentiment**: Analysts hold a **Strong Buy** consensus on Amazon stocks, with price targets suggesting a potential increase of 20-40% over the next year. However, some analysts caution about the sustainability of this bullish outlook amid rising economic uncertainties and competition in the AI sector.

4. **Investment in AI**: Amazon is heavily investing in AI and cloud capabilities, positioning itself strategically for future growth. The company has aggressively entered the bond market to fund these initiatives.

5. **Regulatory Environment**: Legal challenges and potential regulatory changes in the EU seem to be easing, which may positively impact Amazon's operational flexibility.

6. **Future Outlook**: Analysts generally see Amazon as undervalued by about 25-35% over the next year, contingent on the company's ability to convert its massive AI investments into sustainable high-margin revenues without excessively inflating debt levels.

Overall, while Amazon's immediate stock performance shows promise driven by AWS growth and strategic investments, careful scrutiny of their AI expenditures and broader economic factors will be essential for maintaining investor confidence.

# NEW SECTION: Introducing polygon.io

In [10]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [11]:
from polygon import RESTClient

client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=276.97, high=280.38, low=275.25, open=275.27, timestamp=1764104400000, volume=46824959.0, vwap=277.9957)

In [13]:
from market import get_share_price

get_share_price("AAPL")

276.97

In [14]:
for i in range(1000):
    get_share_price("AAPL")

In [15]:
get_share_price("AAPL")

276.97

In [16]:
from agents.mcp.server import MCPServerStdio

params = {"command": "uv", "args": ["run", "market_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='lookup_share_price', title=None, description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'lookup_share_priceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

In [17]:
from agents import Agent, Runner
from IPython.display import Markdown, display

instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4o-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The share price of Apple (AAPL) is $276.97.